# 03 · Land maps from AlphaEarth (main method, step 2)

A random forest on AlphaEarth 2024 embeddings, trained on official labels (CDL; NWI wetlands in North Dakota) in
the **training tiles of every area**, maps the land classes of the **test tiles** (North Dakota, North Carolina)
and of the **held-out states** (Minnesota, Georgia). Solar farms are taught from just **10 farms** chosen at random
(seed 0) among the training tiles' farms built by 2023; the other training farms' pixels are left out of training.

Also here (fixed in `PLAN.md`, amendment A2):
- **K, shuffled labels:** the same classifier trained with shuffled labels must show no skill;
- **D0 inputs:** maps from **planted** vectors (each pixel's official class, one-hot, plus noise), scored in
  notebook 04;
- **X1, X2:** the solar-farm map against USPVDB in the North Carolina test tiles and in the held-out states.

In [1]:
import json
import time

import numpy as np
import pandas as pd

from askmap.config import PROJECT_ROOT, data_root, load_config
from askmap.data import read_uspvdb
from askmap.layers import SOLAR_EXISTING, SOLAR_LATER, official_leaf_map
from askmap.models import fit_classifier, predict_map, training_sample
from askmap.score import f1
from askmap.tiles import area_tiles, solar_farm_ids, solar_raster

cfg = load_config(PROJECT_ROOT / "configs" / "feasibility.yaml")
ROOT = data_root()
RES = PROJECT_ROOT / "outputs" / "results"
MAPS = ROOT / "maps"
MAPS.mkdir(exist_ok=True)
CL, MC = cfg["classes"], cfg["classifier"]
LEAVES = CL["leaves"]
OTHER = LEAVES.index("other")
SOLAR = LEAVES.index("solar_farm")
LAST = cfg["uspvdb"]["last_year"]
T_START = time.time()
pv = read_uspvdb(ROOT / "raw" / "uspvdbGeoJSON.zip", cfg["uspvdb"]["member"])
T = {}
for area, acfg in cfg["areas"].items():
    for name, (tile, role) in area_tiles(acfg, cfg["grid"]).items():
        z = np.load(ROOT / "tiles" / f"{area}_{name}.npz")
        solar = z["solar"] if "solar" in z.files else None
        nwi = z["nwi"] if "nwi" in z.files else None
        ae = z["ae"].astype(np.float32)
        valid = np.isfinite(ae).all(axis=0)
        if solar is not None:
            valid &= solar != SOLAR_LATER
        T[name] = {
            "area": area,
            "role": role,
            "tile": tile,
            "ae": ae,
            "valid": valid,
            "solar": solar,
            "leaf": official_leaf_map(z["cdl"], LEAVES, CL, nwi=nwi, solar=solar),
        }
TRAIN = [n for n, t in T.items() if t["role"] == "train"]
TEST = [n for n, t in T.items() if t["role"] == "test"]
print(f"training tiles {TRAIN}\ntest tiles {TEST}")

training tiles ['Stutsman', 'Barnes', 'Wells', 'NC1', 'NC2', 'NC3', 'SD1', 'SD2', 'NE1', 'NE2', 'KS1', 'KS2', 'VA1', 'VA2', 'VA3', 'SC1', 'SC2']
test tiles ['Kidder', 'Foster', 'Eddy', 'NC4', 'NC5', 'NC6', 'MN1', 'MN2', 'MN3', 'GA1', 'GA2', 'GA3']


## The 10 solar farms used for training

In [2]:
rng = np.random.default_rng(MC["seed"])
farm_ids = sorted({i for n in TRAIN for i in solar_farm_ids(pv, T[n]["tile"], LAST)})
handful = sorted(rng.choice(farm_ids, size=MC["solar_handful"], replace=False).tolist())
pv_hand = pv[pv.case_id.isin(handful)]
keep = {}
for n in TRAIN:
    k = T[n]["valid"].copy()
    if T[n]["solar"] is not None:
        hand = solar_raster(pv_hand, T[n]["tile"], LAST) == SOLAR_EXISTING
        k &= ~((T[n]["solar"] == SOLAR_EXISTING) & ~hand)
    keep[n] = k
hand_px = int(sum(((T[n]["leaf"] == SOLAR) & keep[n]).sum() for n in TRAIN))
print(f"training-tile farms built by {LAST}: {len(farm_ids)}; handful {handful}; their pixels: {hand_px}")
pv_hand[["case_id", "p_state", "p_year", "p_area"]]

training-tile farms built by 2023: 58; handful [400143, 400589, 400786, 401123, 401220, 401373, 401937, 402411, 403526, 404556]; their pixels: 1076


,case_id,p_state,p_year,p_area
988,403526,NC,2018,62580
2052,401123,NC,2017,207487
2555,401220,NC,2015,87876
2634,402411,SC,2017,292709
2700,400786,NC,2013,67944
2926,404556,SC,2022,155025
4125,400589,NC,2013,31344
5489,401373,NC,2016,35368
5548,400143,NC,2014,21178
5558,401937,NC,2015,107660


## Train the classifier and map the test tiles

In [3]:
def fit_and_map(features, labels_shuffled=False, tiles=TEST):
    X, y = training_sample(
        [features[n] for n in TRAIN],
        [T[n]["leaf"] for n in TRAIN],
        [keep[n] for n in TRAIN],
        len(LEAVES),
        MC["per_class"],
        MC["seed"],
    )
    if labels_shuffled:
        y = np.random.default_rng(MC["seed"]).permutation(y)
    rf = fit_classifier(X, y, MC["trees"], MC["min_samples_leaf"], MC["seed"])
    counts = dict(zip(np.array(LEAVES)[np.unique(y)], np.bincount(y)[np.unique(y)].tolist(), strict=True))
    return {n: predict_map(rf, features[n], OTHER) for n in tiles}, counts


t0 = time.time()
AE = {n: T[n]["ae"] for n in T}
pred, counts = fit_and_map(AE)
for n, m in pred.items():
    np.savez_compressed(MAPS / f"ae_{n}.npz", leaf=m)
print(f"AlphaEarth classifier: {time.time() - t0:.0f}s; training pixels per class {counts}")

AlphaEarth classifier: 23s; training pixels per class {np.str_('corn'): 20000, np.str_('soybeans'): 20000, np.str_('spring_wheat'): 20000, np.str_('sunflower'): 1697, np.str_('canola'): 4715, np.str_('cotton'): 20000, np.str_('other_crop'): 20000, np.str_('hay'): 20000, np.str_('grassland'): 20000, np.str_('fallow'): 20000, np.str_('forest'): 20000, np.str_('developed'): 20000, np.str_('water'): 20000, np.str_('wetland'): 20000, np.str_('solar_farm'): 1076, np.str_('other'): 20000}


## How well the AlphaEarth maps match the official maps, class by class (information)

In [4]:
def class_table(maps, tiles):
    rows = []
    for c, name in enumerate(LEAVES):
        g = np.concatenate([(T[n]["leaf"] == c)[T[n]["valid"]] for n in tiles])
        p = np.concatenate([(maps[n] == c)[T[n]["valid"]] for n in tiles])
        rows.append({"class": name, "true_share": g.mean(), "predicted_share": p.mean(), "F1": f1(p, g)})
    return pd.DataFrame(rows)


POOLS = {
    a: [n for n in TEST if T[n]["area"] == a]
    for a in ("north_dakota", "north_carolina", "minnesota", "georgia")
}
ct = pd.concat([class_table(pred, tiles).assign(pool=a) for a, tiles in POOLS.items()])
ct.to_csv(RES / "03_class_f1.csv", index=False)
ct.pivot(index="class", columns="pool", values="F1").round(3)

pool,georgia,minnesota,north_carolina,north_dakota
class,,,,
canola,0.000,0.000,1.000,0.853
corn,0.788,0.918,0.840,0.924
cotton,0.779,1.000,0.701,1.000
developed,0.755,0.748,0.742,0.434
fallow,0.000,0.179,0.010,0.324
forest,0.679,0.845,0.752,0.391
grassland,0.536,0.615,0.242,0.834
hay,0.200,0.523,0.144,0.665
other,0.089,0.135,0.217,0.373


## X1 and X2: solar farms learned from 10 examples

In [5]:
def solar_f1(maps, tiles):
    g = np.concatenate([(T[n]["leaf"] == SOLAR)[T[n]["valid"]] for n in tiles])
    p = np.concatenate([(maps[n] == SOLAR)[T[n]["valid"]] for n in tiles])
    return f1(p, g), int(g.sum()), int(p.sum())


x1 = solar_f1(pred, POOLS["north_carolina"])
x2 = solar_f1(pred, POOLS["minnesota"] + POOLS["georgia"])
X = {
    "X1": {"pass": x1[0] >= 0.50, "F1": x1[0], "true_px": x1[1], "pred_px": x1[2]},
    "X2": {"pass": x2[0] >= 0.40, "F1": x2[0], "true_px": x2[1], "pred_px": x2[2]},
    "by_pool": {a: solar_f1(pred, tiles) for a, tiles in POOLS.items()},
}
for k in ("X1", "X2"):
    print(k, "PASS" if X[k]["pass"] else "FAIL", {kk: vv for kk, vv in X[k].items() if kk != "pass"})

X1 FAIL {'F1': 0.46685082872928174, 'true_px': 6955, 'pred_px': 2457}
X2 FAIL {'F1': 0.31754842440011566, 'true_px': 14414, 'pred_px': 2881}


## K: the same classifier with shuffled training labels (seen test tiles)

In [6]:
t0 = time.time()
seen = POOLS["north_dakota"] + POOLS["north_carolina"]
pred_k, _ = fit_and_map(AE, labels_shuffled=True, tiles=seen)
kt = class_table(pred_k, seen)
p, q = kt.true_share, kt.predicted_share
kt["chance_F1"] = np.where(p + q > 0, 2 * p * q / (p + q).where(p + q > 0, 1), 0)
kt["excess"] = kt.F1 - kt.chance_F1
present = kt.true_share > 0
K = {
    "pass": float(kt.excess[present].mean()) <= 0.02,
    "mean_excess": float(kt.excess[present].mean()),
    "seconds": round(time.time() - t0),
}
print("K", "PASS" if K["pass"] else "FAIL", K)
kt.round(4)

K PASS {'pass': True, 'mean_excess': -0.0004525763975325005, 'seconds': 36}


,class,true_share,predicted_share,F1,chance_F1,excess
0,corn,0.0757,0.0841,0.0784,0.0797,-0.0013
1,soybeans,0.1663,0.0781,0.0833,0.1063,-0.0230
2,spring_wheat,0.0225,0.0795,0.0445,0.0351,0.0094
3,sunflower,0.0005,0.0000,0.0000,0.0000,0.0000
4,canola,0.0020,0.0000,0.0000,0.0000,-0.0000
5,cotton,0.0276,0.0748,0.0504,0.0403,0.0101
6,other_crop,0.0529,0.0742,0.0604,0.0618,-0.0014
7,hay,0.0082,0.0731,0.0126,0.0147,-0.0020
8,grassland,0.2025,0.0800,0.1481,0.1147,0.0334
9,fallow,0.0032,0.0753,0.0074,0.0062,0.0012


## D0 inputs: maps from planted vectors (official class one-hot plus noise)

In [7]:
t0 = time.time()
noise = np.random.default_rng(MC["seed"])
PLANT = {}
for n in TRAIN + TEST:
    onehot = np.eye(len(LEAVES), dtype=np.float32)[T[n]["leaf"]].transpose(2, 0, 1)
    PLANT[n] = np.concatenate(
        [
            onehot + 0.3 * noise.standard_normal(onehot.shape, dtype=np.float32),
            noise.standard_normal((16, *onehot.shape[1:]), dtype=np.float32),
        ]
    )
pred_p, _ = fit_and_map(PLANT)
for n, m in pred_p.items():
    np.savez_compressed(MAPS / f"planted_{n}.npz", leaf=m)
print(
    f"planted maps in {time.time() - t0:.0f}s; class F1 in the seen test tiles (median over classes):",
    round(class_table(pred_p, seen).F1.median(), 3),
)

planted maps in 47s; class F1 in the seen test tiles (median over classes): 0.881


In [8]:
out = {
    "X": X,
    "K": K,
    "handful": handful,
    "training_farms": len(farm_ids),
    "handful_pixels": hand_px,
    "training_pixels_per_class": counts,
    "minutes": round((time.time() - T_START) / 60, 1),
    "maps_MB": round(sum(p.stat().st_size for p in MAPS.iterdir()) / 1e6, 1),
}
(RES / "03_land_maps.json").write_text(json.dumps(out, indent=1, default=float))
print(f"Run time {out['minutes']} min")

Run time 1.9 min
